<a href="https://colab.research.google.com/github/cloudatlasx/vanna_colab/blob/main/vannn_fastapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 安装 Vanna 及 Gemini 相关依赖
这里安装支持 Gemini 和 Flask 后端的 vanna 扩展包。

In [ ]:
!pip install 'vanna[gemini,flask]' --upgrade

# 2. 下载示例数据库

In [ ]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# 3. 导入模块
注意：此处已将 AnthropicLlmService 替换为 GeminiLlmService。

In [ ]:
from vanna import Agent, AgentConfig
from vanna.servers.fastapi import VannaFastAPIServer
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.google import GeminiLlmService
from vanna.tools import RunSqlTool, VisualizeDataTool
from vanna.integrations.sqlite import SqliteRunner
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool
from vanna.integrations.local.agent_memory import DemoAgentMemory
from google.colab import userdata

# 4. Define your User Authentication
Here we're going to say that if you're logged in as `admin@example.com` then you're in the `admin` group, otherwise you're in the `user` group

In [ ]:
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        # In production, validate cookies/JWTs here
        user_email = request_context.get_cookie('vanna_email')
        # 如果是在 Colab 调试环境下没有 Cookie，自动 fallback 到管理员
        if not user_email:
            return User(id="admin_dev", email="admin@example.com", group_memberships=['admin'])

        print(f"Resolving user for email: {user_email}")

        if user_email == "admin@example.com":
            return User(id="admin1", email=user_email, group_memberships=['admin'])

        return User(id="user1", email=user_email, group_memberships=['user'])

# 5. Define the Tools and Access Control

In [ ]:
tools = ToolRegistry()
tools.register_local_tool(RunSqlTool(sql_runner=SqliteRunner(database_path="./Chinook.sqlite")), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])
agent_memory = DemoAgentMemory(max_items=1000)
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])

6. 配置 LLM (Gemini 2.5 Flash)
请确保在 Colab 左侧“钥匙”图标处设置了 `GEMINI_KEY`。

In [ ]:

# 1. 获取 Secret Key
api_key = userdata.get('GEMINI_KEY')

# 2. 初始化 Gemini 2.5 Flash
llm = GeminiLlmService(
    model='gemini-2.5-flash',
    api_key=api_key
)
# 3. 创建 Agent
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    config=AgentConfig(),
    agent_memory=agent_memory
)

# 4. 运行服务器
server = VannaFastAPIServer(agent)
print("Gemini 2.5 Flash 配置完成，准备运行...")
server.run()